In [1]:
from pathlib import Path
from typing import List, Dict
import re
import json

from llama_cpp import Llama


In [2]:
from pathlib import Path

MODELS_DIR = Path(r"C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4\models")

models = {
    "LARGE": MODELS_DIR / "deepseek-coder-6.7b-instruct.Q4_K_M.gguf",
    "MEDIUM": MODELS_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    "SMALL": MODELS_DIR / "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
}

for name, path in models.items():
    print(name, "exists:", path.exists(), "|", path)


LARGE exists: True | C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4\models\deepseek-coder-6.7b-instruct.Q4_K_M.gguf
MEDIUM exists: True | C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4\models\mistral-7b-instruct-v0.2.Q4_K_M.gguf
SMALL exists: True | C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4\models\tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [3]:
llm_large = Llama(
    model_path=str(models["LARGE"]),
    n_ctx=2048,
    n_threads=8,
    n_batch=256,
    verbose=True
)

llm_medium = Llama(
    model_path=str(models["MEDIUM"]),
    n_ctx=2048,
    n_threads=8,
    n_batch=256,
    verbose=True
)

llm_small = Llama(
    model_path=str(models["SMALL"]),
    n_ctx=1024,
    n_threads=8,
    n_batch=128,
    verbose=True
)

print("✅ All Architect models loaded")


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 
AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


✅ All Architect models loaded


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


In [4]:
SQL_PROMPT = """
You are an expert SQL engineer.

STRICT RULES:
- Use ONLY the provided schema
- Do NOT invent tables or columns
- Output ONLY ONE SQL query
- No explanations, no markdown
- Must start with SELECT

Schema:
{schema}

Question:
{question}

SQL:
SELECT
"""


In [5]:
def generate_sql(llm: Llama, schema: str, question: str) -> str | None:
    prompt = SQL_PROMPT.format(schema=schema, question=question)

    out = llm(
        prompt,
        max_tokens=256,
        stop=[";", "\n\n"]
    )

    text = out["choices"][0]["text"].strip()

    if not text:
        return None

    sql = "SELECT " + text
    sql = re.sub(r"\s+", " ", sql).strip()

    if not sql.lower().startswith("select"):
        return None

    return sql + ";"


In [6]:
def classify_intent(sql: str) -> str:
    sql_l = sql.lower()
    if any(k in sql_l for k in ["insert", "update", "delete", "create", "drop", "alter"]):
        return "WRITE"
    return "READ"


In [7]:
def infer_output_shape(sql: str) -> str:
    sql_l = sql.lower()

    if any(k in sql_l for k in ["count(", "max(", "min(", "avg(", "sum("]):
        return "single-row"

    if "group by" in sql_l:
        return "multi-row"

    return "multi-row"


In [8]:
def validate_sql(sql: str, schema: str) -> bool:
    schema_tables = set(re.findall(r"-\s*(\w+)\(", schema))
    used_tables = set(re.findall(r"from\s+(\w+)|join\s+(\w+)", sql.lower()))

    used_tables = {t for pair in used_tables for t in pair if t}

    return used_tables.issubset(schema_tables)



In [9]:
def architect_ensemble(
    question: str,
    schema: str,
    max_candidates: int = 5
) -> list[dict]:

    raw_sqls = []

    sql = generate_sql(llm_large, schema, question)
    if sql:
        raw_sqls.append(sql)

    for _ in range(2):
        sql = generate_sql(llm_medium, schema, question)
        if sql:
            raw_sqls.append(sql)

    raw_sqls = list(dict.fromkeys(raw_sqls))[:max_candidates]

    results = []
    for sql in raw_sqls:
        if not validate_sql(sql, schema):
            continue

        results.append({
            "sql": sql,
            "intent": classify_intent(sql),
            "expected_shape": infer_output_shape(sql)
        })

    return results


In [10]:
schema_text = """
tables:
- student(student_id, name, age)
- course(course_id, title)
- enrollment(student_id, course_id)
"""

question = "List student names and course titles they are enrolled in"

candidates = architect_ensemble(question, schema_text)

print("Generated SQL candidates:\n")
for c in candidates:
    print(json.dumps(c, indent=2))


Llama.generate: prefix-match hit


Generated SQL candidates:

{
  "sql": "SELECT ```sql -- Write your SQL query here ``` ```;",
  "intent": "READ",
  "expected_shape": "multi-row"
}
{
  "sql": "SELECT s.name, c.title FROM student s JOIN enrollment e ON s.student_id = e.student_id JOIN course c ON e.course_id = c.course_id;",
  "intent": "READ",
  "expected_shape": "multi-row"
}


In [11]:
from typing import TypedDict

class ArchitectCandidate(TypedDict):
    sql: str
    intent: str
    expected_shape: str
